# NB52 — 63k Genis Veri: Baseline & Hipotez Ablasyonlari (H1-H8)

Plan: `docs/PLAN_63K_ENTEGRASYON.md` ADIM 3 — **planin en yuksek getirili adimi**.

Eski projenin (NB12-NB48, anonim yarisma verisi) H1-H8 hipotezlerinin 63k'ya (legacy,
acik isimli, buyuk n, dusuk floor) tasinip tasinmadigini **tek-eksen ardisik ablasyon**
ile test eder. Her adimda bir onceki adimin kazanani SABITLENIR, sadece bir sonraki
eksen degisir (NB44/NB48 disiplini). **Test seti bu notebook'ta da HALA acilmaz** --
tum kararlar 5-fold stratified CV'den. Ek olarak her adimda **GroupKFold(base__hugo)
gen-holdout** F1'i de raporlanir (NB51'de fark kucuk cikti ama izlenmeye devam).

| # | Eksen | Kollar | Hipotez |
|---|---|---|---|
| A1 | Model ailesi | LGBM/XGBoost/CatBoost/RandomForest/BalancedBagging | H1, H8 |
| A2 | Missing stratejisi | M1/M3/M5/native_nan | H2 |
| A3 | Sinif dengeleme | yok/class_weight/scale_pos_weight/gercek resample | H3 |
| A4 | Feature seti | tumu/top-200/top-100 (xgb-importance) | H7 |
| A5 | Etiket kalitesi | tumu/qualified-disiplanmis/label_conf-agirlikli/sadece-expert-panel | yeni |
| A6 | Meta-predictor | dahil/haric | NB51'in A6 on-bulgusunun resmilestirilmesi |
| A7 | Feature engineering | no_fe/with_fe (Grantham/BLOSUM62/delta-fizikokimya) | H6 |

In [1]:
# Cell 1: Imports & Config
import os, sys, json, time, warnings
import torch  # ONCE torch import et -- macOS'ta OpenMP/BLAS init sirasi SIGSEGV'ini onlemek icin (dogrulandi: nb52 smoke test)
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

PROJECT_ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, PROJECT_ROOT)

from config import SEED
from src import columns_63k as C63
from src.features import GRANTHAM, BLOSUM62
from src.metrics import optimize_threshold, compute_all_metrics

np.random.seed(SEED)

from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.metrics import f1_score, roc_auc_score, matthews_corrcoef
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from imblearn.ensemble import BalancedBaggingClassifier
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

PARQUET_DIR = os.path.join(PROJECT_ROOT, 'data', '63k_genis')
RESULTS_PREP_DIR = os.path.join(PROJECT_ROOT, 'results', 'v31_63k_prep')
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results', 'v33_63k_baseline')
REPORTS_DIR = os.path.join(PROJECT_ROOT, 'reports')
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

MISSENSE_PARQUET = os.path.join(PARQUET_DIR, 'missense_63k.parquet')
df = pd.read_parquet(MISSENSE_PARQUET)
print('missense_63k.parquet yuklendi:', df.shape)

with open(os.path.join(RESULTS_PREP_DIR, 'nb50_column_lists.json')) as f:
    col_lists = json.load(f)

meta_score_cols = [c for c in col_lists['meta_score_cols'] if c in df.columns]
meta_rankscore_cols = [c for c in col_lists.get('meta_rankscore_cols', []) if c in df.columns]
meta_pred_cols = [c for c in col_lists['meta_pred_cols'] if c in df.columns]
all_meta_cols = list(set(meta_score_cols) | set(meta_rankscore_cols) | set(meta_pred_cols))

numeric_cols = [c for c in col_lists['numeric_cols'] if c in df.columns]
categorical_cols = [c for c in col_lists['categorical_cols'] if c in df.columns]
binary_cols = [c for c in col_lists['binary_cols'] if c in df.columns]
feature_cols_base = numeric_cols + categorical_cols + binary_cols

y = df['Label'].astype(int)
groups_gene = df[C63.GENE_GROUP_COL]
prevalence = y.mean()
FLOOR_F1 = C63.floor_f1(prevalence)
MAJORITY_ACC = max(prevalence, 1 - prevalence)
print(f'n={len(df)}, prevalans={prevalence:.4f}, floor-F1={FLOOR_F1:.4f}, majority-acc={MAJORITY_ACC:.4f}')
print(f'Feature adaylari: numeric={len(numeric_cols)}, categorical={len(categorical_cols)}, meta(ayri)={len(all_meta_cols)}')

RESULTS_LOG = []  # her ablasyon satiri buraya eklenir -> en sonda tek tablo

missense_63k.parquet yuklendi: (60970, 533)
n=60970, prevalans=0.3727, floor-F1=0.5430, majority-acc=0.6273
Feature adaylari: numeric=358, categorical=164, meta(ayri)=63


## Ortak Degerlendirme Altyapisi

Tum ablasyon adimlari ayni `evaluate_recipe()` fonksiyonunu cagirir: 5-fold stratified
CV (model secimi icin) + GroupKFold(base__hugo) (gen-holdout icin, ayni foldlar
uzerinden degil ayri bir CV turu olarak) + train metrikleri (overfit gap) + floor-F1
karsilastirmasi. Boylece her ablasyon satiri ayni sutunlara sahip olur.

In [2]:
# Cell 2: Ortak degerlendirme fonksiyonu
def build_model(family, cat_cols=None, scale_pos_weight=None, class_weight=None):
    if family == 'lgbm':
        params = dict(n_estimators=300, learning_rate=0.05, random_state=SEED, verbosity=-1)
        if scale_pos_weight is not None:
            params['scale_pos_weight'] = scale_pos_weight
        if class_weight == 'balanced':
            params['class_weight'] = 'balanced'
        return lgb.LGBMClassifier(**params)
    if family == 'xgboost':
        params = dict(n_estimators=300, learning_rate=0.05, random_state=SEED,
                       eval_metric='logloss', verbosity=0)
        if scale_pos_weight is not None:
            params['scale_pos_weight'] = scale_pos_weight
        return xgb.XGBClassifier(**params)
    if family == 'catboost':
        params = dict(n_estimators=300, learning_rate=0.05, random_state=SEED, verbose=False)
        if cat_cols:
            params['cat_features'] = cat_cols
        if scale_pos_weight is not None:
            params['scale_pos_weight'] = scale_pos_weight
        return CatBoostClassifier(**params)
    if family == 'random_forest':
        params = dict(n_estimators=300, random_state=SEED, n_jobs=-1)
        if class_weight == 'balanced':
            params['class_weight'] = 'balanced'
        return RandomForestClassifier(**params)
    if family == 'balanced_bagging':
        return BalancedBaggingClassifier(
            estimator=lgb.LGBMClassifier(n_estimators=150, learning_rate=0.05, random_state=SEED, verbosity=-1, n_jobs=1),
            n_estimators=10, random_state=SEED, n_jobs=1,  # n_jobs=-1 macOS'ta LGBM ile ic ice paralellik deadlock'una yol aciyordu (nb52 smoke test)
        )
    raise ValueError(family)


def encode_categoricals_train_only(X_tr, X_val, cat_cols):
    from sklearn.preprocessing import OrdinalEncoder
    if not cat_cols:
        return X_tr, X_val
    X_tr = X_tr.copy(); X_val = X_val.copy()
    enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    X_tr[cat_cols] = X_tr[cat_cols].astype(str).fillna('__NA__')
    X_val[cat_cols] = X_val[cat_cols].astype(str).fillna('__NA__')
    X_tr[cat_cols] = enc.fit_transform(X_tr[cat_cols])
    X_val[cat_cols] = enc.transform(X_val[cat_cols])
    return X_tr, X_val


def stringify_categoricals(X_tr, X_val, cat_cols):
    """CatBoost cat_features icin: NaN'lari '__NA__' string'ine cevirir (CatBoost
    kategorik sutunda ciplak NaN kabul etmiyor -- string bekliyor)."""
    if not cat_cols:
        return X_tr, X_val
    X_tr = X_tr.copy(); X_val = X_val.copy()
    X_tr[cat_cols] = X_tr[cat_cols].astype(str).fillna('__NA__')
    X_val[cat_cols] = X_val[cat_cols].astype(str).fillna('__NA__')
    return X_tr, X_val


def impute_train_only(X_tr, X_val, numeric_subset):
    if not numeric_subset:
        return X_tr, X_val
    # keep_empty_features=True: CV fold icinde bir sutun train'de tamamen NaN kalabilir
    # (nadir/gen-spesifik annotator, ozellikle kucuk alt-orneklerde veya top-100/200
    # feature azaltmasindan sonra). Varsayilan davranis boyle sutunlari SESSIZCE
    # DUSURUR -- bu X_tr'nin beklenen sutun sayisindan az donmesine ve asagi akan
    # 'Columns must be same length as key' hatasina yol acar. keep_empty_features
    # sutunu 0 ile doldurup korur, sutun sayisini sabit tutar.
    imputer = SimpleImputer(strategy='median', keep_empty_features=True)
    X_tr = X_tr.copy(); X_val = X_val.copy()
    X_tr[numeric_subset] = imputer.fit_transform(X_tr[numeric_subset])
    X_val[numeric_subset] = imputer.transform(X_val[numeric_subset])
    return X_tr, X_val


def evaluate_recipe(name, cols, cat_cols, family, y=y, resample_fn=None,
                     scale_pos_weight=None, class_weight=None, n_splits=5,
                     needs_impute=True, sample_weight_col=None, extra_frame=None):
    """
    Tek bir ablasyon kolunu 5-fold stratified CV + GroupKFold(gen) ile degerlendirir.
    resample_fn(X_tr, y_tr) -> (X_tr_res, y_tr_res): A3'un gercek-resample kolu icin.
    sample_weight_col: A5'in label_conf agirlikli kolu icin (df'ten okunur, train-only kullanilir).
    Doner: dict (cv_f1, cv_f1_std, cv_mcc, cv_aucpr, train_f1, gap, floor_f1, genegroup_f1, fit_seconds)
    """
    frame = extra_frame if extra_frame is not None else df
    numeric_subset = [c for c in cols if c in numeric_cols or c not in categorical_cols]
    numeric_subset = [c for c in numeric_subset if c not in cat_cols]

    def _run(splitter, split_iter):
        f1s, mccs, aucprs, train_f1s = [], [], [], []
        t0 = time.time()
        for train_idx, val_idx in split_iter:
            X_tr = frame.iloc[train_idx][cols].copy()
            X_val = frame.iloc[val_idx][cols].copy()
            y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

            if needs_impute and family != 'catboost':
                X_tr, X_val = impute_train_only(X_tr, X_val, numeric_subset)
            if family in ('xgboost', 'random_forest', 'balanced_bagging'):
                X_tr, X_val = encode_categoricals_train_only(X_tr, X_val, cat_cols)
                eff_cat_cols = []
            elif family == 'catboost':
                # CatBoost numeric NaN'i native destekler ama kategorik NaN'i DESTEKLEMEZ --
                # ciplak NaN kategorik hucre gorunce ArrowInvalid/CatBoostError firlatir.
                X_tr, X_val = stringify_categoricals(X_tr, X_val, cat_cols)
                eff_cat_cols = cat_cols
            else:
                eff_cat_cols = cat_cols

            sw_tr = None
            if sample_weight_col is not None:
                sw_tr = frame.iloc[train_idx][sample_weight_col].values

            if resample_fn is not None:
                X_tr, y_tr = resample_fn(X_tr, y_tr)
                sw_tr = None  # resample sonrasi orijinal index hizasi bozulur

            model = build_model(family, cat_cols=eff_cat_cols, scale_pos_weight=scale_pos_weight, class_weight=class_weight)
            fit_kwargs = {}
            if sw_tr is not None and family in ('lgbm', 'xgboost'):
                fit_kwargs['sample_weight'] = sw_tr

            model.fit(X_tr, y_tr, **fit_kwargs) if fit_kwargs else model.fit(X_tr, y_tr)

            proba_val = model.predict_proba(X_val)[:, 1]
            pred_val = (proba_val >= 0.5).astype(int)
            f1s.append(f1_score(y_val, pred_val))
            mccs.append(matthews_corrcoef(y_val, pred_val))
            from sklearn.metrics import average_precision_score
            aucprs.append(average_precision_score(y_val, proba_val))

            proba_tr = model.predict_proba(X_tr)[:, 1]
            train_f1s.append(f1_score(y_tr, (proba_tr >= 0.5).astype(int)))
        elapsed = time.time() - t0
        return np.array(f1s), np.array(mccs), np.array(aucprs), np.array(train_f1s), elapsed

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    f1s, mccs, aucprs, train_f1s, elapsed = _run(skf, skf.split(frame, y))

    gkf = GroupKFold(n_splits=n_splits)
    f1s_group, _, _, _, _ = _run(gkf, gkf.split(frame, y, groups=groups_gene.iloc[frame.index] if hasattr(groups_gene, 'iloc') else groups_gene))

    result = {
        'name': name, 'n_features': len(cols),
        'cv_f1': float(f1s.mean()), 'cv_f1_std': float(f1s.std()),
        'cv_mcc': float(mccs.mean()), 'cv_aucpr': float(aucprs.mean()),
        'train_f1': float(train_f1s.mean()),
        'gap': float(train_f1s.mean() - f1s.mean()),
        'floor_f1': float(FLOOR_F1),
        'genegroup_f1': float(f1s_group.mean()),
        'fit_seconds': round(elapsed, 1),
    }
    RESULTS_LOG.append(result)
    print(f"[{name}] cv_f1={result['cv_f1']:.4f}+/-{result['cv_f1_std']:.4f} "
          f"mcc={result['cv_mcc']:.4f} aucpr={result['cv_aucpr']:.4f} "
          f"gap={result['gap']:.4f} gene_f1={result['genegroup_f1']:.4f} "
          f"({result['fit_seconds']}s)")
    return result

## A1 — Model Ailesi (H1, H8)

Meta-predictor DAHIL, tum feature'lar, M3 missing (varsayilan baslangic -- flag hicbir
kolda henuz eklenmedi, SimpleImputer zaten NaN'lari medyanla dolduruyor; bu adimda flag
YOK, sadece A2'de eklenecek). Kazanan model ailesi A2'den itibaren sabitlenir.

In [3]:
# Cell 3: A1 -- Model ailesi
cat_cols_all = [c for c in categorical_cols if c in feature_cols_base]

a1_results = {}
for family in ['lgbm', 'xgboost', 'catboost', 'random_forest', 'balanced_bagging']:
    res = evaluate_recipe(f'A1_{family}', feature_cols_base, cat_cols_all, family)
    a1_results[family] = res

best_a1_family = max(a1_results, key=lambda k: a1_results[k]['cv_f1'])
print()
print(f'A1 KAZANAN: {best_a1_family} (cv_f1={a1_results[best_a1_family]["cv_f1"]:.4f})')

h1_verdict = 'DOGRULANDI' if best_a1_family == 'balanced_bagging' else 'CURUTULDU'
h8_note = 'NN/DNN bu turda test edilmedi (agac aileleri once taranir, NN A1 sonrasi ayrica denenir)'
print(f'H1 (BalancedBagging en guclu) -> {h1_verdict}')

[A1_lgbm] cv_f1=0.9897+/-0.0006 mcc=0.9837 aucpr=0.9993 gap=0.0103 gene_f1=0.9881 (52.5s)
[A1_xgboost] cv_f1=0.9894+/-0.0009 mcc=0.9832 aucpr=0.9992 gap=0.0105 gene_f1=0.9883 (142.1s)
[A1_catboost] cv_f1=0.9888+/-0.0007 mcc=0.9821 aucpr=0.9991 gap=0.0027 gene_f1=0.9876 (505.6s)
[A1_random_forest] cv_f1=0.9854+/-0.0007 mcc=0.9766 aucpr=0.9984 gap=0.0146 gene_f1=0.9843 (141.1s)
[A1_balanced_bagging] cv_f1=0.9882+/-0.0009 mcc=0.9811 aucpr=0.9990 gap=0.0070 gene_f1=0.9873 (334.6s)

A1 KAZANAN: lgbm (cv_f1=0.9897)
H1 (BalancedBagging en guclu) -> CURUTULDU


### H8 kontrolu — SmallMLP (NN), A1'in en iyi agac ailesiyle karsilastirma

NB39'un aksine burada n=61k (21x daha buyuk havuz) -- H8 "NN artik rekabetci olmali"
diyordu. Basit bir MLP (BatchNorm yok, dropout+early stopping, focal loss) ile tek
bir 5-fold CV turu.

In [4]:
# Cell 4: H8 -- SmallMLP karsilastirmasi
import torch
import torch.nn as nn
from src.focal_loss import FocalLoss

class SmallMLP(nn.Module):
    def __init__(self, n_in, hidden=128, dropout=0.4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden // 2, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)


def run_cv_mlp(X_full, y_full, cat_cols, n_splits=5, max_epochs=60, patience=8):
    # NN icin kategorikleri ordinal + tum sutunlari standardize ediyoruz (train-only fit)
    from sklearn.preprocessing import StandardScaler
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    f1s = []
    for train_idx, val_idx in skf.split(X_full, y_full):
        X_tr = X_full.iloc[train_idx].copy(); X_val = X_full.iloc[val_idx].copy()
        y_tr, y_val = y_full.iloc[train_idx].values, y_full.iloc[val_idx].values

        X_tr, X_val = impute_train_only(X_tr, X_val, [c for c in X_full.columns if c not in cat_cols])
        X_tr, X_val = encode_categoricals_train_only(X_tr, X_val, cat_cols)

        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_tr.values)
        X_val_s = scaler.transform(X_val.values)

        # Train icinde kucuk bir val-split early stopping icin
        n_val_es = max(200, int(0.1 * len(X_tr_s)))
        X_tr_fit, X_tr_es = X_tr_s[:-n_val_es], X_tr_s[-n_val_es:]
        y_tr_fit, y_tr_es = y_tr[:-n_val_es], y_tr[-n_val_es:]

        model = SmallMLP(X_tr_s.shape[1])
        opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
        loss_fn = FocalLoss(alpha=0.25, gamma=2.0)

        Xt = torch.tensor(X_tr_fit, dtype=torch.float32)
        yt = torch.tensor(y_tr_fit, dtype=torch.float32)
        Xes = torch.tensor(X_tr_es, dtype=torch.float32)
        yes = torch.tensor(y_tr_es, dtype=torch.float32)

        best_es_loss, patience_ctr, best_state = np.inf, 0, None
        for epoch in range(max_epochs):
            model.train()
            perm = torch.randperm(len(Xt))
            for i in range(0, len(Xt), 512):
                idx = perm[i:i+512]
                opt.zero_grad()
                out = model(Xt[idx])
                loss = loss_fn(out, yt[idx])
                loss.backward()
                opt.step()
            model.eval()
            with torch.no_grad():
                es_loss = loss_fn(model(Xes), yes).item()
            if es_loss < best_es_loss - 1e-4:
                best_es_loss, patience_ctr = es_loss, 0
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
            else:
                patience_ctr += 1
                if patience_ctr >= patience:
                    break
        if best_state is not None:
            model.load_state_dict(best_state)

        model.eval()
        with torch.no_grad():
            proba_val = torch.sigmoid(model(torch.tensor(X_val_s, dtype=torch.float32))).numpy()
        pred_val = (proba_val >= 0.5).astype(int)
        f1s.append(f1_score(y_val, pred_val))
    return np.array(f1s)

X_mlp = df[feature_cols_base].copy()
f1s_mlp = run_cv_mlp(X_mlp, y, cat_cols_all)
mlp_f1 = float(f1s_mlp.mean())
RESULTS_LOG.append({
    'name': 'A1_smallmlp_H8', 'n_features': len(feature_cols_base),
    'cv_f1': mlp_f1, 'cv_f1_std': float(f1s_mlp.std()),
    'cv_mcc': None, 'cv_aucpr': None, 'train_f1': None, 'gap': None,
    'floor_f1': float(FLOOR_F1), 'genegroup_f1': None, 'fit_seconds': None,
})
print(f'SmallMLP cv_f1={mlp_f1:.4f}+/-{f1s_mlp.std():.4f} (karsilastirma: {best_a1_family}={a1_results[best_a1_family]["cv_f1"]:.4f})')

h8_verdict = 'DOGRULANDI (NN rekabetci)' if mlp_f1 >= a1_results[best_a1_family]['cv_f1'] - 0.01 else 'CURUTULDU (agaclar hala onde)'
print(f'H8 (NN artik rekabetci) -> {h8_verdict}')

# A1'in nihai kazanani: agac aileleri + MLP arasindan
A1_WINNER = best_a1_family if a1_results[best_a1_family]['cv_f1'] >= mlp_f1 else 'smallmlp'
print(f'A1 NIHAI KAZANAN (A2+ icin sabitlenecek): {A1_WINNER}')

SmallMLP cv_f1=0.9836+/-0.0011 (karsilastirma: lgbm=0.9897)
H8 (NN artik rekabetci) -> DOGRULANDI (NN rekabetci)
A1 NIHAI KAZANAN (A2+ icin sabitlenecek): lgbm


## A2 — Missing Stratejisi (H2)

A1 kazanani sabit. M1 (flag yok+medyan) / M3 (flag+medyan, mevcut varsayilan) /
M5 (flag+secici drop-veya-medyan) / native_nan (impute yok, agaca birak -- sadece
LGBM/CatBoost/XGBoost native NaN destekliyorsa anlamli).

In [5]:
# Cell 5: A2 -- Missing stratejisi (A1 kazanani sabit)
FAMILY = A1_WINNER if A1_WINNER != 'smallmlp' else best_a1_family  # MLP kazanirsa bile A2+ agac ailesiyle devam (imputation eksenini agacta izole etmek icin)
print(f'A2+ icin sabit model ailesi: {FAMILY}')

missing_pct = df[numeric_cols].isna().mean()
high_missing_cols = missing_pct[missing_pct > 0.5].index.tolist()
low_missing_cols = [c for c in numeric_cols if c not in high_missing_cols]
print(f'>%50 eksik numeric sutun: {len(high_missing_cols)}, <=%50: {len(low_missing_cols)}')

df_m = df.copy()
for c in high_missing_cols:
    df_m[f'is_missing_{c}'] = df_m[c].isna().astype(int)
flag_cols = [f'is_missing_{c}' for c in high_missing_cols]

# M1: flag yok, tum numeric medyan impute (zaten evaluate_recipe'in varsayilani)
m1_cols = feature_cols_base
res_m1 = evaluate_recipe('A2_M1_no_flag', m1_cols, cat_cols_all, FAMILY, extra_frame=df_m)

# M3: flag (>%50 NaN) + tum numeric medyan impute (orijinal sutun korunur)
m3_cols = feature_cols_base + flag_cols
res_m3 = evaluate_recipe('A2_M3_flag_median', m3_cols, cat_cols_all, FAMILY, extra_frame=df_m)

# M5: flag + >%50 NaN sutunlarini DROP et, <=%50 medyan impute
m5_cols = [c for c in feature_cols_base if c not in high_missing_cols] + flag_cols
res_m5 = evaluate_recipe('A2_M5_flag_selective_drop', m5_cols, cat_cols_all, FAMILY, extra_frame=df_m)

# native_nan: impute yok (sadece lgbm/xgboost/catboost native NaN destekler)
if FAMILY in ('lgbm', 'xgboost', 'catboost'):
    res_native = evaluate_recipe('A2_native_nan', feature_cols_base, cat_cols_all, FAMILY,
                                   extra_frame=df_m, needs_impute=False)
else:
    res_native = None
    print('native_nan atlandi (secili model ailesi native NaN desteklemiyor)')

a2_candidates = {'M1': res_m1, 'M3': res_m3, 'M5': res_m5}
if res_native is not None:
    a2_candidates['native_nan'] = res_native
best_a2_key = max(a2_candidates, key=lambda k: a2_candidates[k]['cv_f1'])
print()
print(f'A2 KAZANAN: {best_a2_key} (cv_f1={a2_candidates[best_a2_key]["cv_f1"]:.4f})')
h2_verdict = 'DOGRULANDI' if best_a2_key == 'M5' else 'CURUTULDU/DEGISTI'
print(f'H2 (M5 >= M3) -> {h2_verdict}')

A2_WINNER_COLS = {'M1': m1_cols, 'M3': m3_cols, 'M5': m5_cols, 'native_nan': feature_cols_base}[best_a2_key]
A2_NEEDS_IMPUTE = best_a2_key != 'native_nan'
A2_FRAME = df_m

A2+ icin sabit model ailesi: lgbm
>%50 eksik numeric sutun: 116, <=%50: 242
[A2_M1_no_flag] cv_f1=0.9897+/-0.0006 mcc=0.9837 aucpr=0.9993 gap=0.0103 gene_f1=0.9881 (65.0s)
[A2_M3_flag_median] cv_f1=0.9896+/-0.0009 mcc=0.9834 aucpr=0.9993 gap=0.0104 gene_f1=0.9882 (69.3s)
[A2_M5_flag_selective_drop] cv_f1=0.9898+/-0.0006 mcc=0.9837 aucpr=0.9993 gap=0.0102 gene_f1=0.9881 (57.8s)
[A2_native_nan] cv_f1=0.9900+/-0.0008 mcc=0.9841 aucpr=0.9993 gap=0.0100 gene_f1=0.9885 (65.2s)

A2 KAZANAN: native_nan (cv_f1=0.9900)
H2 (M5 >= M3) -> CURUTULDU/DEGISTI


## A3 — Sinif Dengeleme (H3)

A1+A2 kazananlari sabit. yok (baseline) / class_weight=balanced / scale_pos_weight /
gercek balanced resample (undersample cogunluk sinifi train-fold icinde).

In [6]:
# Cell 6: A3 -- Sinif dengeleme
def resample_balanced(X_tr, y_tr):
    """Cogunluk sinifini (benign, Label=0) azinliga (patho, Label=1) esitleyecek sekilde undersample."""
    idx_pos = y_tr[y_tr == 1].index
    idx_neg = y_tr[y_tr == 0].index
    n = min(len(idx_pos), len(idx_neg))
    rng = np.random.RandomState(SEED)
    idx_pos_s = rng.choice(idx_pos, size=n, replace=False)
    idx_neg_s = rng.choice(idx_neg, size=n, replace=False)
    idx_all = np.concatenate([idx_pos_s, idx_neg_s])
    rng.shuffle(idx_all)
    return X_tr.loc[idx_all], y_tr.loc[idx_all]

spw = (y == 0).sum() / (y == 1).sum()
print(f'scale_pos_weight (neg/pos orani) = {spw:.4f}')

res_none = evaluate_recipe('A3_none', A2_WINNER_COLS, cat_cols_all, FAMILY,
                            extra_frame=A2_FRAME, needs_impute=A2_NEEDS_IMPUTE)
res_cw = evaluate_recipe('A3_class_weight', A2_WINNER_COLS, cat_cols_all, FAMILY,
                          extra_frame=A2_FRAME, needs_impute=A2_NEEDS_IMPUTE, class_weight='balanced')
res_spw = evaluate_recipe('A3_scale_pos_weight', A2_WINNER_COLS, cat_cols_all, FAMILY,
                           extra_frame=A2_FRAME, needs_impute=A2_NEEDS_IMPUTE, scale_pos_weight=spw)
res_resample = evaluate_recipe('A3_real_resample', A2_WINNER_COLS, cat_cols_all, FAMILY,
                                extra_frame=A2_FRAME, needs_impute=A2_NEEDS_IMPUTE, resample_fn=resample_balanced)

a3_candidates = {'none': res_none, 'class_weight': res_cw, 'scale_pos_weight': res_spw, 'real_resample': res_resample}
best_a3_key = max(a3_candidates, key=lambda k: a3_candidates[k]['cv_f1'])
print()
print(f'A3 KAZANAN: {best_a3_key} (cv_f1={a3_candidates[best_a3_key]["cv_f1"]:.4f})')
h3_verdict = 'DOGRULANDI' if best_a3_key == 'real_resample' else 'CURUTULDU/DEGISTI'
print(f'H3 (gercek resample > class_weight) -> {h3_verdict}')

A3_KWARGS = {
    'none': {},
    'class_weight': {'class_weight': 'balanced'},
    'scale_pos_weight': {'scale_pos_weight': spw},
    'real_resample': {'resample_fn': resample_balanced},
}[best_a3_key]

scale_pos_weight (neg/pos orani) = 1.6833
[A3_none] cv_f1=0.9900+/-0.0008 mcc=0.9841 aucpr=0.9993 gap=0.0100 gene_f1=0.9885 (55.3s)
[A3_class_weight] cv_f1=0.9900+/-0.0006 mcc=0.9840 aucpr=0.9993 gap=0.0100 gene_f1=0.9884 (54.4s)
[A3_scale_pos_weight] cv_f1=0.9903+/-0.0011 mcc=0.9845 aucpr=0.9993 gap=0.0097 gene_f1=0.9883 (56.6s)
[A3_real_resample] cv_f1=0.9891+/-0.0007 mcc=0.9825 aucpr=0.9992 gap=0.0109 gene_f1=0.9878 (52.0s)

A3 KAZANAN: scale_pos_weight (cv_f1=0.9903)
H3 (gercek resample > class_weight) -> CURUTULDU/DEGISTI


## A4 — Feature Seti (H7)

A1+A2+A3 kazananlari sabit. tumu / xgb-importance top-200 / top-100.

In [7]:
# Cell 7: A4 -- Feature seti (xgb-importance siralamasi)
X_imp = A2_FRAME[A2_WINNER_COLS].copy()
cat_in_a2 = [c for c in cat_cols_all if c in A2_WINNER_COLS]
X_imp, _ = encode_categoricals_train_only(X_imp, X_imp.iloc[:1].copy(), cat_in_a2)
X_imp, _ = impute_train_only(X_imp, X_imp.iloc[:1].copy(), [c for c in A2_WINNER_COLS if c not in cat_in_a2])

xgb_imp_model = xgb.XGBClassifier(n_estimators=300, learning_rate=0.05, random_state=SEED, eval_metric='logloss', verbosity=0)
xgb_imp_model.fit(X_imp, y)
importance_a4 = pd.Series(xgb_imp_model.feature_importances_, index=A2_WINNER_COLS).sort_values(ascending=False)

top200_cols = importance_a4.head(200).index.tolist()
top100_cols = importance_a4.head(100).index.tolist()

res_all_feat = evaluate_recipe('A4_all_features', A2_WINNER_COLS, cat_cols_all, FAMILY,
                                extra_frame=A2_FRAME, needs_impute=A2_NEEDS_IMPUTE, **A3_KWARGS)
res_top200 = evaluate_recipe('A4_top200', top200_cols, [c for c in cat_cols_all if c in top200_cols], FAMILY,
                              extra_frame=A2_FRAME, needs_impute=A2_NEEDS_IMPUTE, **A3_KWARGS)
res_top100 = evaluate_recipe('A4_top100', top100_cols, [c for c in cat_cols_all if c in top100_cols], FAMILY,
                              extra_frame=A2_FRAME, needs_impute=A2_NEEDS_IMPUTE, **A3_KWARGS)

a4_candidates = {'all': res_all_feat, 'top200': res_top200, 'top100': res_top100}
best_a4_key = max(a4_candidates, key=lambda k: a4_candidates[k]['cv_f1'])
print()
print(f'A4 KAZANAN: {best_a4_key} (cv_f1={a4_candidates[best_a4_key]["cv_f1"]:.4f})')
h7_verdict = 'DOGRULANDI' if best_a4_key in ('top200', 'top100') else 'CURUTULDU'
print(f'H7 (feature selection kazandirir) -> {h7_verdict}')

A4_WINNER_COLS = {'all': A2_WINNER_COLS, 'top200': top200_cols, 'top100': top100_cols}[best_a4_key]
A4_CAT_COLS = [c for c in cat_cols_all if c in A4_WINNER_COLS]

[A4_all_features] cv_f1=0.9903+/-0.0011 mcc=0.9845 aucpr=0.9993 gap=0.0097 gene_f1=0.9883 (54.2s)
[A4_top200] cv_f1=0.9904+/-0.0009 mcc=0.9848 aucpr=0.9993 gap=0.0096 gene_f1=0.9886 (31.8s)
[A4_top100] cv_f1=0.9900+/-0.0007 mcc=0.9841 aucpr=0.9993 gap=0.0100 gene_f1=0.9883 (14.9s)

A4 KAZANAN: top200 (cv_f1=0.9904)
H7 (feature selection kazandirir) -> DOGRULANDI


## A5 — Etiket Kalitesi (63k'ya ozgu yeni eksen)

A1-A4 kazananlari sabit. tum etiketler / `Label_qualified` disiplanmis / `label_conf`
agirlikli (sample_weight) / yalnizca expert-panel (label_conf==1.0).

In [8]:
# Cell 8: A5 -- Etiket kalitesi
res_a5_all = evaluate_recipe('A5_all_labels', A4_WINNER_COLS, A4_CAT_COLS, FAMILY,
                              extra_frame=A2_FRAME, needs_impute=A2_NEEDS_IMPUTE, **A3_KWARGS)

df_qualified_excl = A2_FRAME[~A2_FRAME['Label_qualified']].reset_index(drop=True)
y_qualified_excl = df_qualified_excl['Label'].astype(int)
groups_gene_bak = groups_gene
groups_gene = df_qualified_excl[C63.GENE_GROUP_COL]  # evaluate_recipe genegroup icin global okuyor
res_a5_qualified_excl = evaluate_recipe('A5_qualified_excluded', A4_WINNER_COLS, A4_CAT_COLS, FAMILY,
                                         y=y_qualified_excl, extra_frame=df_qualified_excl,
                                         needs_impute=A2_NEEDS_IMPUTE, **A3_KWARGS)
groups_gene = groups_gene_bak

res_a5_weighted = evaluate_recipe('A5_label_conf_weighted', A4_WINNER_COLS, A4_CAT_COLS, FAMILY,
                                   extra_frame=A2_FRAME, needs_impute=A2_NEEDS_IMPUTE,
                                   sample_weight_col='label_conf', **A3_KWARGS)

df_expert_only = A2_FRAME[A2_FRAME['label_conf'] >= 1.0].reset_index(drop=True)
if len(df_expert_only) > 200 and df_expert_only['Label'].nunique() == 2:
    y_expert = df_expert_only['Label'].astype(int)
    groups_gene_bak = groups_gene
    groups_gene = df_expert_only[C63.GENE_GROUP_COL]
    res_a5_expert = evaluate_recipe('A5_expert_panel_only', A4_WINNER_COLS, A4_CAT_COLS, FAMILY,
                                     y=y_expert, extra_frame=df_expert_only,
                                     needs_impute=A2_NEEDS_IMPUTE, **A3_KWARGS)
    groups_gene = groups_gene_bak
else:
    res_a5_expert = None
    print(f'expert_panel_only atlandi (n={len(df_expert_only)}, yetersiz veya tek sinif)')

a5_candidates = {'all_labels': res_a5_all, 'qualified_excluded': res_a5_qualified_excl, 'label_conf_weighted': res_a5_weighted}
if res_a5_expert is not None:
    a5_candidates['expert_panel_only'] = res_a5_expert
best_a5_key = max(a5_candidates, key=lambda k: a5_candidates[k]['cv_f1'])
print()
print(f'A5 KAZANAN: {best_a5_key} (cv_f1={a5_candidates[best_a5_key]["cv_f1"]:.4f})')
print('NOT: expert_panel_only ve qualified_excluded farkli n uzerinde CV yaptigi icin cv_f1 dogrudan kiyaslanabilir DEGIL (farkli test havuzu) -- sadece all_labels ile karsilastirma anlamli, digerleri bilgi amacli.')

[A5_all_labels] cv_f1=0.9904+/-0.0009 mcc=0.9848 aucpr=0.9993 gap=0.0096 gene_f1=0.9886 (31.3s)
[A5_qualified_excluded] cv_f1=0.9903+/-0.0009 mcc=0.9846 aucpr=0.9994 gap=0.0097 gene_f1=0.9886 (30.4s)
[A5_label_conf_weighted] cv_f1=0.9902+/-0.0010 mcc=0.9843 aucpr=0.9993 gap=0.0098 gene_f1=0.9884 (30.2s)
[A5_expert_panel_only] cv_f1=0.9852+/-0.0021 mcc=0.9534 aucpr=0.9988 gap=0.0148 gene_f1=0.9831 (10.9s)

A5 KAZANAN: all_labels (cv_f1=0.9904)
NOT: expert_panel_only ve qualified_excluded farkli n uzerinde CV yaptigi icin cv_f1 dogrudan kiyaslanabilir DEGIL (farkli test havuzu) -- sadece all_labels ile karsilastirma anlamli, digerleri bilgi amacli.


## A6 — Meta-Predictor Ablasyonu (NB51'in on-bulgusunun resmilestirilmesi)

A1-A4 kazananlari sabit (A5 sadece bilgi amacli, ana hatta A5_all_labels devam eder).
NB51'de meta-predictor'lar tamamen cikarilinca CV F1 sadece 0.0021 dustu -- burada
A4'un secili feature seti uzerinde teyit edilir.

In [9]:
# Cell 9: A6 -- Meta-predictor dahil/haric (A4 feature seti icinde)
meta_in_a4 = [c for c in A4_WINNER_COLS if c in all_meta_cols]
nometa_a4_cols = [c for c in A4_WINNER_COLS if c not in all_meta_cols]
print(f'A4 feature setinde meta-predictor sayisi: {len(meta_in_a4)} / {len(A4_WINNER_COLS)}')

res_a6_with = evaluate_recipe('A6_with_meta', A4_WINNER_COLS, A4_CAT_COLS, FAMILY,
                               extra_frame=A2_FRAME, needs_impute=A2_NEEDS_IMPUTE, **A3_KWARGS)
res_a6_without = evaluate_recipe('A6_without_meta', nometa_a4_cols, [c for c in A4_CAT_COLS if c in nometa_a4_cols], FAMILY,
                                  extra_frame=A2_FRAME, needs_impute=A2_NEEDS_IMPUTE, **A3_KWARGS)

a6_delta = res_a6_with['cv_f1'] - res_a6_without['cv_f1']
print()
print(f'A6 fark (dahil-haric) = {a6_delta:.4f}')
a6_decision = 'DAHIL ET (fark>0.05, modelin isini yapiyor ama performans kaybi buyuk)' if a6_delta > 0.05 else 'DAHIL ET (fark<=0.05, dusuk risk, NB51 ile tutarli)'
print(f'A6 KARAR: {a6_decision}')

A6_WINNER_COLS = A4_WINNER_COLS if res_a6_with['cv_f1'] >= res_a6_without['cv_f1'] else nometa_a4_cols
A6_CAT_COLS = [c for c in A4_CAT_COLS if c in A6_WINNER_COLS]

A4 feature setinde meta-predictor sayisi: 36 / 200
[A6_with_meta] cv_f1=0.9904+/-0.0009 mcc=0.9848 aucpr=0.9993 gap=0.0096 gene_f1=0.9886 (34.3s)
[A6_without_meta] cv_f1=0.9867+/-0.0007 mcc=0.9787 aucpr=0.9989 gap=0.0133 gene_f1=0.9796 (28.1s)

A6 fark (dahil-haric) = 0.0038
A6 KARAR: DAHIL ET (fark<=0.05, dusuk risk, NB51 ile tutarli)


## A7 — Feature Engineering (H6)

A1-A4-A6 kazananlari sabit. no_fe (baseline) / with_fe (Grantham/BLOSUM62/delta-
hidropati/hacim/pI/yuk, `base__achange`'den turetilmis -- `src/columns_63k.py::
compute_achange_fe`, `src/features.py::GRANTHAM/BLOSUM62` yeniden kullanildi).

In [10]:
# Cell 10: A7 -- Feature Engineering ablasyonu
achange_fe = C63.compute_achange_fe(df[C63.ACHANGE_COL], GRANTHAM, BLOSUM62)
fe_numeric_cols = ['grantham', 'blosum62', 'delta_hydropathy', 'delta_volume', 'delta_pi', 'delta_charge']
fe_binary_cols = ['is_stopgain', 'is_synonymous']
fe_cols_all = fe_numeric_cols + fe_binary_cols

df_fe = A2_FRAME.copy()
for c in fe_cols_all:
    df_fe[c] = achange_fe[c].values

res_a7_no_fe = evaluate_recipe('A7_no_fe', A6_WINNER_COLS, A6_CAT_COLS, FAMILY,
                                extra_frame=A2_FRAME, needs_impute=A2_NEEDS_IMPUTE, **A3_KWARGS)
with_fe_cols = A6_WINNER_COLS + fe_cols_all
res_a7_with_fe = evaluate_recipe('A7_with_fe', with_fe_cols, A6_CAT_COLS, FAMILY,
                                  extra_frame=df_fe, needs_impute=True, **A3_KWARGS)

a7_delta = res_a7_with_fe['cv_f1'] - res_a7_no_fe['cv_f1']
print()
print(f'A7 fark (with_fe - no_fe) = {a7_delta:.4f}')
h6_verdict = 'FE FAYDALI' if a7_delta > 0.005 else ('FE ZARARLI' if a7_delta < -0.005 else 'FE NOTR')
print(f'H6 (FE panel-bagimli, korukoruye acma) -> {h6_verdict}')

A7_WINNER_COLS = with_fe_cols if a7_delta > 0 else A6_WINNER_COLS
FINAL_FRAME = df_fe if a7_delta > 0 else A2_FRAME

[A7_no_fe] cv_f1=0.9904+/-0.0009 mcc=0.9848 aucpr=0.9993 gap=0.0096 gene_f1=0.9886 (31.9s)
[A7_with_fe] cv_f1=0.9897+/-0.0006 mcc=0.9836 aucpr=0.9993 gap=0.0103 gene_f1=0.9880 (31.7s)

A7 fark (with_fe - no_fe) = -0.0007
H6 (FE panel-bagimli, korukoruye acma) -> FE NOTR


## H1-H8 Taşınma Karnesi + 63k Champion Reçetesi

In [11]:
# Cell 11: Ozet tablo + H1-H8 karnesi + champion recete kaydi
results_df = pd.DataFrame(RESULTS_LOG)
print(results_df.to_string(index=False))
results_df.to_csv(os.path.join(RESULTS_DIR, 'nb52_ablation_results.csv'), index=False)

hypothesis_scorecard = {
    'H1_balbag_strongest': h1_verdict,
    'H2_m5_gte_m3': h2_verdict,
    'H3_real_resample_gt_classweight': h3_verdict,
    'H4_meta_lr_not_gbm': 'NB53de test edilecek (stacking asamasi)',
    'H5_heterogeneous_base_corr_lt_085': 'NB53de test edilecek (stacking asamasi)',
    'H6_fe_panel_dependent': h6_verdict,
    'H7_feature_selection_helps': h7_verdict,
    'H8_nn_competitive_at_scale': h8_verdict,
}
print()
print('=== H1-H8 TASINMA KARNESI ===')
for k, v in hypothesis_scorecard.items():
    print(f'{k}: {v}')

with open(os.path.join(RESULTS_DIR, 'nb52_hypothesis_scorecard.json'), 'w') as f:
    json.dump(hypothesis_scorecard, f, indent=2, ensure_ascii=False)

champion_recipe = {
    'model_family': FAMILY,
    'missing_strategy': best_a2_key,
    'class_balancing': best_a3_key,
    'feature_set': best_a4_key,
    'n_features_final': len(A7_WINNER_COLS),
    'meta_predictor': 'included' if A6_WINNER_COLS == A4_WINNER_COLS else 'excluded',
    'feature_engineering': 'with_fe' if a7_delta > 0 else 'no_fe',
    'cv_f1_final': RESULTS_LOG[-1]['cv_f1'],
    'cv_mcc_final': RESULTS_LOG[-1]['cv_mcc'],
    'genegroup_f1_final': RESULTS_LOG[-1]['genegroup_f1'],
    'floor_f1': float(FLOOR_F1),
}
print()
print('=== 63K CHAMPION RECETESI (tek model, NB53 icin baslangic noktasi) ===')
for k, v in champion_recipe.items():
    print(f'{k}: {v}')

with open(os.path.join(RESULTS_DIR, 'nb52_champion_recipe.json'), 'w') as f:
    json.dump(champion_recipe, f, indent=2)

with open(os.path.join(RESULTS_DIR, 'nb52_champion_feature_list.json'), 'w') as f:
    json.dump({'feature_cols': A7_WINNER_COLS, 'cat_cols': A6_CAT_COLS}, f, indent=2)

                     name  n_features    cv_f1  cv_f1_std   cv_mcc  cv_aucpr  train_f1      gap  floor_f1  genegroup_f1  fit_seconds
                  A1_lgbm         522 0.989748   0.000603 0.983656  0.999279  1.000000 0.010252  0.542991      0.988140         52.5
               A1_xgboost         522 0.989447   0.000889 0.983170  0.999165  0.999934 0.010487  0.542991      0.988254        142.1
              A1_catboost         522 0.988758   0.000742 0.982078  0.999120  0.991480 0.002721  0.542991      0.987574        505.6
         A1_random_forest         522 0.985356   0.000655 0.976647  0.998449  1.000000 0.014644  0.542991      0.984266        141.1
      A1_balanced_bagging         522 0.988152   0.000938 0.981092  0.999046  0.995129 0.006977  0.542991      0.987306        334.6
           A1_smallmlp_H8         522 0.983639   0.001144      NaN       NaN       NaN      NaN  0.542991           NaN          NaN
            A2_M1_no_flag         522 0.989748   0.000603 0.983656  0

In [12]:
# Cell 12: Gorsel -- ablasyon ilerlemesi (her eksende kazananin cv_f1'i)
progress_points = []
for key, cand in [('A1', a1_results[best_a1_family]), ('A2', a2_candidates[best_a2_key]),
                   ('A3', a3_candidates[best_a3_key]), ('A4', a4_candidates[best_a4_key]),
                   ('A6', res_a6_with if A6_WINNER_COLS == A4_WINNER_COLS else res_a6_without),
                   ('A7', res_a7_with_fe if a7_delta > 0 else res_a7_no_fe)]:
    progress_points.append((key, cand['cv_f1']))

fig, ax = plt.subplots(figsize=(8, 5))
xs = [p[0] for p in progress_points]
ys = [p[1] for p in progress_points]
ax.plot(xs, ys, marker='o', linewidth=2)
ax.axhline(FLOOR_F1, color='red', linestyle='--', label=f'Floor-F1={FLOOR_F1:.3f}')
ax.set_ylabel('CV F1 (pathogenic)')
ax.set_title('NB52 Ablasyon Ilerlemesi (her eksenin kazanani)')
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, 'nb52_ablation_progress.png'), dpi=120)
plt.close(fig)
print('Grafik kaydedildi: nb52_ablation_progress.png')

Grafik kaydedildi: nb52_ablation_progress.png


## Otomatik PDF Rapor

In [13]:
# Cell 13: PDF rapor
from fpdf import FPDF

class NB52Report(FPDF):
    def header(self):
        self.set_font('Helvetica', 'B', 14)
        self.cell(0, 10, 'NB52 - Baseline & Hipotez Ablasyonlari (H1-H8) Raporu', ln=True, align='C')
        self.ln(2)

    def section(self, title):
        self.set_font('Helvetica', 'B', 12)
        self.cell(0, 8, title, ln=True)
        self.set_font('Helvetica', '', 9)

    def kv_table(self, d):
        for k, v in d.items():
            self.cell(0, 6, f'{k}: {v}', ln=True)
        self.ln(2)

report = NB52Report()
report.add_page()

report.section('1. Veri ve Floor')
report.kv_table({'n': len(df), 'prevalans': f'{prevalence:.4f}', 'floor_f1': f'{FLOOR_F1:.4f}'})

report.section('2. H1-H8 Tasinma Karnesi')
report.kv_table(hypothesis_scorecard)

report.section('3. 63k Champion Recetesi')
report.kv_table(champion_recipe)

report.section('4. Tum Ablasyon Sonuclari (ozet)')
for row in RESULTS_LOG:
    line = f"{row['name']}: cv_f1={row['cv_f1']:.4f}" if row['cv_f1'] is not None else f"{row['name']}: -"
    report.cell(0, 5, line, ln=True)

REPORT_PATH = os.path.join(REPORTS_DIR, 'nb52_baseline_report.pdf')
report.output(REPORT_PATH)
print(f'PDF rapor yazildi: {REPORT_PATH}')

PDF rapor yazildi: /Users/tefe/teknofest_model/teknofest_model/reports/nb52_baseline_report.pdf


## Sonraki Adim

**ADIM 4 — NB53 (`notebooks/53_63k_optimization.ipynb`):** Champion reçeteye Optuna
(TRIALS_TREE=100 / TRIALS_NN=50, `config.py`'dan), stacking (H4: LR vs GBM meta, H5:
heterojen base pairwise korelasyon <0.85), kalibrasyon (Platt/Isotonic/Venn-Abers +
ECE), threshold finalizasyonu (F1-max vs MCC-max), non-missense ek-veri ablasyonu.
Test seti hâlâ açılmadı — NB54'e kadar.